In [1]:
from pathlib import Path
import re
import unicodedata
import csv
import pandas as pd
import numpy as np

# Dependencias necesarias en el entorno:
# %pip install openpyxl pyarrow


In [2]:
DATA_DIR = Path(r"C:\Users\Home\Documents\Datos Ebsa")
OUT_DIR = DATA_DIR / "Procesado"
DETAIL_DIR = OUT_DIR / "detalle_mensual"

OUT_DIR.mkdir(parents=True, exist_ok=True)
DETAIL_DIR.mkdir(parents=True, exist_ok=True)


EXTENSIONES_VALIDAS = {".xlsx", ".csv"}

archivos = sorted(
    archivo
    for archivo in DATA_DIR.iterdir()
    if archivo.is_file()
    and archivo.suffix.lower() in EXTENSIONES_VALIDAS
    and not archivo.name.startswith("~$")
)

print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos:
    print(f"{archivo.suffix.lower():<6} | {archivo.name}")
print(f"Archivos encontrados: {len(archivos)}")

for archivo in archivos[:10]:
    print(archivo.name)


Archivos encontrados: 12
.xlsx  | formato_tc2_20221.xlsx
.xlsx  | formato_tc2_202210.xlsx
.xlsx  | formato_tc2_202211.xlsx
.xlsx  | formato_tc2_202212.xlsx
.xlsx  | formato_tc2_20222.xlsx
.csv   | formato_tc2_20223.csv
.xlsx  | formato_tc2_20224.xlsx
.xlsx  | formato_tc2_20225.xlsx
.xlsx  | formato_tc2_20226.xlsx
.xlsx  | formato_tc2_20227.xlsx
.xlsx  | formato_tc2_20228.xlsx
.xlsx  | formato_tc2_20229.xlsx
Archivos encontrados: 12
formato_tc2_20221.xlsx
formato_tc2_202210.xlsx
formato_tc2_202211.xlsx
formato_tc2_202212.xlsx
formato_tc2_20222.xlsx
formato_tc2_20223.csv
formato_tc2_20224.xlsx
formato_tc2_20225.xlsx
formato_tc2_20226.xlsx
formato_tc2_20227.xlsx


In [3]:
# ============================================================
# ESQUEMA CANÓNICO
# ============================================================
# Estos son los nombres internos que usaremos SIEMPRE,
# aunque un archivo de otro año tenga una variante del encabezado.

COLUMNAS_ESENCIALES = [
    "NIU",
    "Tipo Factura",
    "Tipo de Lectura",
    "Consumo Usuario (kWh)",
    "Consumo Promedio Semestral (kWh)",
    "Tipo Medidor",
    "Tarifa Aplicada ($/kWh)"
]


# Estas columnas son útiles pero NO son obligatorias.
# Si no existen, el procesamiento continúa.

COLUMNAS_OPCIONALES = [
    "Año de reporte",
    "Mes de reporte",
    "Estrato",
    "ID Factura",
    "Días Facturados",
    "Fecha de Lectura Actual",
    "Fecha de Lectura Anterior",
    "Refacturación por Consumo Usuario - (kWh)",
    "Ciclo",
    "Clase de Servicio"
]


# Todas son buscadas en los archivos,
# pero solamente COLUMNAS_ESENCIALES provocan error.

COLUMNAS_OBJETIVO = list(
    dict.fromkeys(
        COLUMNAS_ESENCIALES
        + COLUMNAS_OPCIONALES
    )
)


# ============================================================
# ALIAS DE ENCABEZADOS
# ============================================================
# No hacemos fuzzy matching automático porque puede confundir
# columnas parecidas pero con significado distinto, por ejemplo:
# "Estrato" != necesariamente "Estrato TC1"
# "Tarifa Aplicada ($/kWh)" != necesariamente "CU Tarifa"

ALIASES_COLUMNAS = {
    "NIU": [
        "NIU",
        "N.I.U",
    ],
    "Tipo Factura": [
        "Tipo Factura",
        "Tipo de Factura",
    ],
    "Tipo de Lectura": [
        "Tipo de Lectura",
        "Tipo Lectura",
    ],
    "Consumo Usuario (kWh)": [
        "Consumo Usuario (kWh)",
        "Consumo Usuario kWh",
        "Consumo del Usuario (kWh)",
        "Consumo del Usuario kWh",
    ],
    "Consumo Promedio Semestral (kWh)": [
        "Consumo Promedio Semestral (kWh)",
        "Consumo Promedio Semestral kWh",
        "Promedio Semestral Consumo (kWh)",
    ],
    "Tipo Medidor": [
        "Tipo Medidor",
        "Tipo de Medidor",
    ],
    "Tarifa Aplicada ($/kWh)": [
        "Tarifa Aplicada ($/kWh)",
        "Tarifa Aplicada (COP/kWh)",
        "Tarifa Aplicada kWh",
        "Tarifa Aplicada",
    ],
    "Año de reporte": [
        "Año de reporte",
        "Año reporte",
        "Ano de reporte",
        "Ano reporte",
        "Anio de reporte",
        "Anio reporte",
    ],
    "Mes de reporte": [
        "Mes de reporte",
        "Mes reporte",
    ],
    "Estrato": [
        "Estrato",
        "Estrato Socioeconómico",
        "Estrato Socioeconomico",
    ],
    "ID Factura": [
        "ID Factura",
        "Id Factura",
        "ID de Factura",
        "Identificador Factura",
    ],
    "Días Facturados": [
        "Días Facturados",
        "Dias Facturados",
        "Días de Facturación",
        "Dias de Facturacion",
    ],
    "Fecha de Lectura Actual": [
        "Fecha de Lectura Actual",
        "Fecha Lectura Actual",
    ],
    "Fecha de Lectura Anterior": [
        "Fecha de Lectura Anterior",
        "Fecha Lectura Anterior",
    ],
    "Refacturación por Consumo Usuario - (kWh)": [
        "Refacturación por Consumo Usuario - (kWh)",
        "Refacturacion por Consumo Usuario - (kWh)",
        "Refacturación por Consumo Usuario (kWh)",
        "Refacturacion por Consumo Usuario (kWh)",
    ],
    "Ciclo": [
        "Ciclo",
        "Ciclo Facturación",
        "Ciclo Facturacion",
    ],
    "Clase de Servicio": [
        "Clase de Servicio",
        "Clase Servicio",
    ],
}


def normalizar_nombre_columna(nombre):
    """
    Normaliza un encabezado solo para compararlo:
    - elimina tildes
    - pasa a minúsculas
    - elimina saltos de línea
    - homogeneiza signos/puntuación y espacios
    """
    texto = str(nombre).replace("\n", " ").replace("\r", " ").strip()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    texto = texto.lower()
    texto = re.sub(r"[^a-z0-9]+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto


def resolver_columnas(encabezados):
    """
    Busca en un archivo las columnas equivalentes al esquema canónico.

    Retorna:
      - mapeo: {nombre_real_en_excel: nombre_canonico}
      - faltantes: columnas canónicas no encontradas
      - ambiguas: columnas donde más de un encabezado podría coincidir
    """
    encabezados = list(encabezados)

    por_normalizado = {}
    for original in encabezados:
        clave = normalizar_nombre_columna(original)
        por_normalizado.setdefault(clave, []).append(original)

    mapeo = {}
    ambiguas = {}

    for canonica in COLUMNAS_OBJETIVO:
        alias = [canonica] + ALIASES_COLUMNAS.get(canonica, [])
        claves_validas = {
            normalizar_nombre_columna(x)
            for x in alias
        }

        candidatos = []
        for clave in claves_validas:
            candidatos.extend(por_normalizado.get(clave, []))

        # Eliminar duplicados conservando orden
        candidatos = list(dict.fromkeys(candidatos))

        if not candidatos:
            continue

        # Si existe exactamente el nombre canónico, tiene prioridad.
        if canonica in candidatos:
            elegida = canonica
        else:
            elegida = candidatos[0]

        mapeo[elegida] = canonica

        if len(candidatos) > 1:
            ambiguas[canonica] = candidatos

    encontradas = set(mapeo.values())
    faltantes = [
        col for col in COLUMNAS_OBJETIVO
        if col not in encontradas
    ]

    return mapeo, faltantes, ambiguas

# ============================================================
# DETECCIÓN AUTOMÁTICA DE LA HOJA QUE CONTIENE LOS DATOS TC2
# ============================================================

def detectar_hoja_datos_excel(ruta, mostrar=False):
    """
    Revisa todas las hojas de un Excel y selecciona
    automáticamente la que mejor coincide con el esquema TC2.

    La selección se basa en la cantidad de columnas canónicas
    reconocidas por resolver_columnas().
    """

    excel = pd.ExcelFile(
        ruta,
        engine="openpyxl"
    )

    candidatos = []

    for hoja in excel.sheet_names:

        try:

            encabezados = pd.read_excel(
                excel,
                sheet_name=hoja,
                nrows=0
            ).columns.tolist()

            mapeo, faltantes, ambiguas = (
                resolver_columnas(encabezados)
            )

            columnas_reconocidas = len(mapeo)

            # Variables especialmente importantes
            columnas_clave = {
                "NIU",
                "Consumo Usuario (kWh)",
                "Tipo de Lectura",
                "Año de reporte",
                "Mes de reporte"
            }

            reconocidas = set(
                mapeo.values()
            )

            claves_encontradas = len(
                columnas_clave.intersection(
                    reconocidas
                )
            )

            candidatos.append({
                "hoja": hoja,
                "columnas_totales":
                    len(encabezados),
                "columnas_reconocidas":
                    columnas_reconocidas,
                "claves_encontradas":
                    claves_encontradas
            })

        except Exception as e:

            candidatos.append({
                "hoja": hoja,
                "columnas_totales": 0,
                "columnas_reconocidas": 0,
                "claves_encontradas": 0
            })


    if not candidatos:

        raise ValueError(
            f"No se encontraron hojas en {ruta.name}"
        )


    candidatos_df = pd.DataFrame(
        candidatos
    )


    # --------------------------------------------------------
    # Priorizamos:
    # 1. Mayor cantidad de variables clave
    # 2. Mayor cantidad de columnas reconocidas
    # 3. Mayor cantidad total de columnas
    # --------------------------------------------------------

    mejor = (
        candidatos_df
        .sort_values(
            [
                "claves_encontradas",
                "columnas_reconocidas",
                "columnas_totales"
            ],
            ascending=False
        )
        .iloc[0]
    )


    if mostrar:

        print(
            f"\nArchivo: {ruta.name}"
        )

        display(candidatos_df)

        print(
            f"Hoja seleccionada: "
            f"{mejor['hoja']}"
        )


    # Seguridad mínima:
    # una hoja TC2 debería tener NIU y varias
    # columnas reconocidas.
    if (
        mejor["columnas_reconocidas"] < 3
        and
        mejor["claves_encontradas"] == 0
    ):

        raise ValueError(
            f"No se pudo identificar una hoja TC2 "
            f"válida en {ruta.name}"
        )


    return mejor["hoja"]

def extraer_periodo_nombre(ruta):
    """
    Fallback para recuperar año/mes desde nombres como:
      formato_tc2_20221.xlsx
      formato_tc2_202210.xlsx
      formato_tc2_20221 (1).xlsx
    """
    stem = Path(ruta).stem

    match = re.search(
        r"(20\d{2})(1[0-2]|0?[1-9])(?!\d)",
        stem
    )

    if not match:
        return None, None

    return int(match.group(1)), int(match.group(2))


# ============================================================
# AUDITORÍA ROBUSTA DE ESQUEMAS
# EXCEL + CSV
# ============================================================

def auditar_esquemas(archivos):
    
    filas = []
    
    for ruta in archivos:
        
        print(f"Auditando: {ruta.name}")
        
        try:
            
            # ----------------------------------
            # Obtener encabezados
            # ----------------------------------
            
            encabezados = leer_encabezados(ruta)
            
            
            # ----------------------------------
            # Resolver nombres canónicos
            # ----------------------------------
            
            mapeo, faltantes, ambiguas = (
                resolver_columnas(encabezados)
            )
            
            
            # ----------------------------------
            # Periodo desde nombre archivo
            # ----------------------------------
            
            año, mes = extraer_periodo_nombre(ruta)
            
            
            # ----------------------------------
            # Información del CSV
            # ----------------------------------
            
            if ruta.suffix.lower() == ".csv":
                
                config_csv = (
                    detectar_configuracion_csv(ruta)
                )
                
                separador = repr(config_csv["sep"])
                encoding = config_csv["encoding"]
                
            else:
                
                separador = ""
                encoding = ""
            
            
            filas.append({
                
                "archivo": ruta.name,
                "formato": ruta.suffix.lower(),
                "estado": "OK",
                
                "columnas_archivo":
                    len(encabezados),
                
                "columnas_reconocidas":
                    len(mapeo),
                
                "faltantes":
                    ", ".join(faltantes),
                
                "ambiguas":
                    str(ambiguas)
                    if ambiguas
                    else "",
                
                "año_nombre": año,
                "mes_nombre": mes,
                
                "separador_csv": separador,
                "encoding_csv": encoding,
                
                "error": ""
            })
        
        
        except Exception as e:
            
            filas.append({
                
                "archivo": ruta.name,
                "formato": ruta.suffix.lower(),
                "estado": "ERROR",
                
                "columnas_archivo": None,
                "columnas_reconocidas": None,
                
                "faltantes": "",
                "ambiguas": "",
                
                "año_nombre": None,
                "mes_nombre": None,
                
                "separador_csv": "",
                "encoding_csv": "",
                
                "error":
                    f"{type(e).__name__}: {str(e)}"
            })
    
    
    return pd.DataFrame(filas)


In [4]:
# ============================================================
# LEER DATOS
# XLSX / CSV
# ============================================================

def leer_datos(ruta, columnas_reales):

    extension = ruta.suffix.lower()


    # ========================================================
    # EXCEL
    # ========================================================

    if extension == ".xlsx":

        hoja = detectar_hoja_datos_excel(
            ruta
        )

        return pd.read_excel(
            ruta,
            sheet_name=hoja,
            usecols=columnas_reales,
            engine="openpyxl"
        )


    # ========================================================
    # CSV
    # ========================================================

    elif extension == ".csv":

        config = detectar_configuracion_csv(
            ruta
        )

        return pd.read_csv(
            ruta,
            usecols=columnas_reales,
            sep=config["sep"],
            encoding=config["encoding"],
            low_memory=False
        )


    else:

        raise ValueError(
            f"Formato no soportado: {extension}"
        )

In [5]:
# ============================================================
# PROCESAR ARCHIVO
# XLSX / CSV
# ============================================================

def procesar_archivo(
    ruta,
    mostrar_detalle=False
):

    print(
        f"Procesando: {ruta.name} "
        f"[{ruta.suffix.lower()}]"
    )

    # ========================================================
    # 1. OBTENER AÑO Y MES DESDE EL NOMBRE DEL ARCHIVO
    # ========================================================

    año_nombre, mes_nombre = (
        extraer_periodo_nombre(ruta)
    )

    if año_nombre is None or mes_nombre is None:

        raise ValueError(
            f"No fue posible obtener año/mes "
            f"desde el nombre: {ruta.name}"
        )


    # ========================================================
    # 2. LEER ENCABEZADOS
    # ========================================================

    encabezados = leer_encabezados(ruta)


    # ========================================================
    # 3. IDENTIFICAR COLUMNAS
    # ========================================================

    mapeo, faltantes, ambiguas = (
        resolver_columnas(encabezados)
    )


    # ========================================================
    # 4. VALIDAR SOLO COLUMNAS REALMENTE ESENCIALES
    # ========================================================

    faltantes_esenciales = [

        columna

        for columna in COLUMNAS_ESENCIALES

        if columna in faltantes
    ]


    if faltantes_esenciales:

        raise ValueError(

            f"{ruta.name} no contiene "
            f"columnas esenciales: "
            f"{faltantes_esenciales}"
        )


    # ========================================================
    # 5. LEER SOLAMENTE COLUMNAS RECONOCIDAS
    # ========================================================

    columnas_reales = list(
        mapeo.keys()
    )

    df = leer_datos(
        ruta,
        columnas_reales
    )


    # ========================================================
    # 6. RENOMBRAR AL ESQUEMA CANÓNICO
    # ========================================================

    df = df.rename(
        columns=mapeo
    )


    # ========================================================
    # 7. NIU
    # ========================================================

    df["NIU"] = (
        df["NIU"]
        .astype("string")
        .str.strip()
    )


    # Quitar NIU realmente vacíos
    df = df[
        df["NIU"].notna()
        &
        (df["NIU"] != "")
    ].copy()


    # ========================================================
    # 8. GUARDAR AÑO/MES ORIGINALES PARA AUDITORÍA
    # ========================================================
    #
    # No los utilizaremos para construir el periodo,
    # pero conservarlos sirve para descubrir errores
    # dentro de los archivos originales.
    # ========================================================

    if "Año de reporte" in df.columns:

        df["Año_reporte_original"] = (
            pd.to_numeric(
                df["Año de reporte"],
                errors="coerce"
            )
        )


    if "Mes de reporte" in df.columns:

        df["Mes_reporte_original"] = (
            pd.to_numeric(
                df["Mes de reporte"],
                errors="coerce"
            )
        )


    # ========================================================
    # 9. EL NOMBRE DEL ARCHIVO ES LA FUENTE OFICIAL
    # ========================================================
    #
    # Ejemplo:
    #
    # formato_tc2_20225.xlsx
    #
    # SIEMPRE:
    # Año = 2022
    # Mes = 5
    #
    # aunque dentro del archivo existan valores incorrectos.
    # ========================================================

    df["Año de reporte"] = año_nombre

    df["Mes de reporte"] = mes_nombre


    # ========================================================
    # 10. CREAR PERIODO
    # ========================================================

    df["periodo"] = pd.Timestamp(
        year=int(año_nombre),
        month=int(mes_nombre),
        day=1
    )


    # ========================================================
    # 11. CONVERSIÓN DE VARIABLES NUMÉRICAS
    # ========================================================

    columnas_numericas = [

        "Consumo Usuario (kWh)",

        "Consumo Promedio Semestral (kWh)",

        "Tarifa Aplicada ($/kWh)",

        "Días Facturados",

        "Refacturación por Consumo Usuario - (kWh)",

        "Ciclo",

        "Estrato"
    ]


    for columna in columnas_numericas:

        if columna in df.columns:

            df[columna] = pd.to_numeric(
                df[columna],
                errors="coerce"
            )


    # ========================================================
    # 12. FECHAS DE LECTURA
    # ========================================================

    columnas_fecha = [

        "Fecha de Lectura Actual",

        "Fecha de Lectura Anterior"
    ]


    for columna in columnas_fecha:

        if columna in df.columns:

            df[columna] = pd.to_datetime(
                df[columna],
                errors="coerce",
                dayfirst=True
            )


    # ========================================================
    # 13. ESTRATO ES OPCIONAL
    # ========================================================
    #
    # Si el archivo no tiene Estrato,
    # creamos la columna vacía.
    #
    # Así todos los Parquet mantienen
    # una estructura compatible.
    # ========================================================

    if "Estrato" not in df.columns:

        df["Estrato"] = pd.NA


    # ========================================================
    # 14. VARIABLES DE CONTROL
    # ========================================================

    df["archivo_origen"] = ruta.name

    df["formato_origen"] = (
        ruta.suffix.lower()
    )


    # ========================================================
    # 15. DETECTAR INCONSISTENCIAS AÑO/MES
    # ========================================================

    if "Año_reporte_original" in df.columns:

        df["error_año_original"] = (

            df["Año_reporte_original"].notna()

            &

            (
                df["Año_reporte_original"]
                != año_nombre
            )
        )


    if "Mes_reporte_original" in df.columns:

        df["error_mes_original"] = (

            df["Mes_reporte_original"].notna()

            &

            (
                df["Mes_reporte_original"]
                != mes_nombre
            )
        )


    # ========================================================
    # 16. INFORMACIÓN OPCIONAL
    # ========================================================

    if mostrar_detalle:

        print(
            f"    Filas: {len(df):,}"
        )

        print(
            f"    NIU únicos: "
            f"{df['NIU'].nunique():,}"
        )

        print(
            f"    Periodo asignado: "
            f"{año_nombre}-{mes_nombre:02d}"
        )

        print(
            f"    Columnas: "
            f"{len(df.columns)}"
        )


    return df

In [6]:
# ============================================================
# DETECCIÓN AUTOMÁTICA DE CSV
# ============================================================

def detectar_configuracion_csv(ruta):
    
    encodings = [
        "utf-8-sig",
        "utf-8",
        "cp1252",
        "latin1"
    ]
    
    ultimo_error = None
    
    for encoding in encodings:
        
        try:
            
            with open(
                ruta,
                "r",
                encoding=encoding,
                errors="strict"
            ) as f:
                
                muestra = f.read(10000)
            
            # Intentar detectar separador
            try:
                
                dialecto = csv.Sniffer().sniff(
                    muestra,
                    delimiters=",;|\t"
                )
                
                separador = dialecto.delimiter
                
            except csv.Error:
                # El más común en Colombia
                separador = ";"
            
            return {
                "encoding": encoding,
                "sep": separador
            }
        
        except UnicodeDecodeError as e:
            ultimo_error = e
    
    
    raise ValueError(
        f"No se pudo determinar la codificación de {ruta.name}. "
        f"Último error: {ultimo_error}"
    )

In [7]:
# ============================================================
# LEER ENCABEZADOS
# XLSX: DETECTA AUTOMÁTICAMENTE LA HOJA TC2
# CSV: DETECTA SEPARADOR Y ENCODING
# ============================================================

def leer_encabezados(ruta):

    extension = ruta.suffix.lower()


    # ========================================================
    # EXCEL
    # ========================================================

    if extension == ".xlsx":

        hoja = detectar_hoja_datos_excel(
            ruta
        )

        df = pd.read_excel(
            ruta,
            sheet_name=hoja,
            nrows=0,
            engine="openpyxl"
        )

        return df.columns.tolist()


    # ========================================================
    # CSV
    # ========================================================

    elif extension == ".csv":

        config = detectar_configuracion_csv(
            ruta
        )

        df = pd.read_csv(
            ruta,
            nrows=0,
            sep=config["sep"],
            encoding=config["encoding"]
        )

        return df.columns.tolist()


    else:

        raise ValueError(
            f"Formato no soportado: {extension}"
        )

In [8]:
# ============================================================
# AUDITORÍA DE ESQUEMAS ANTES DE CARGAR MILLONES DE REGISTROS
# ============================================================

auditoria = auditar_esquemas(archivos)

display(auditoria)

# Archivos que requieren atención:
display(
    auditoria[
        (auditoria["faltantes"] != "")
        | (auditoria["ambiguas"] != "")
    ]
)


Auditando: formato_tc2_20221.xlsx
Auditando: formato_tc2_202210.xlsx
Auditando: formato_tc2_202211.xlsx
Auditando: formato_tc2_202212.xlsx
Auditando: formato_tc2_20222.xlsx
Auditando: formato_tc2_20223.csv
Auditando: formato_tc2_20224.xlsx
Auditando: formato_tc2_20225.xlsx
Auditando: formato_tc2_20226.xlsx
Auditando: formato_tc2_20227.xlsx
Auditando: formato_tc2_20228.xlsx
Auditando: formato_tc2_20229.xlsx


,archivo,formato,estado,columnas_archivo,columnas_reconocidas,faltantes,ambiguas,año_nombre,mes_nombre,separador_csv,encoding_csv,error
0,formato_tc2_20221.xlsx,.xlsx,OK,72,17,,,2022,1,,,
1,formato_tc2_202210.xlsx,.xlsx,OK,72,17,,,2022,10,,,
2,formato_tc2_202211.xlsx,.xlsx,OK,72,17,,,2022,11,,,
3,formato_tc2_202212.xlsx,.xlsx,OK,72,17,,,2022,12,,,
4,formato_tc2_20222.xlsx,.xlsx,OK,72,17,,,2022,2,,,
5,formato_tc2_20223.csv,.csv,OK,69,11,"Año de reporte, Estrato, Días Facturados, Refa...",,2022,3,"','",cp1252,
6,formato_tc2_20224.xlsx,.xlsx,OK,72,17,,,2022,4,,,
7,formato_tc2_20225.xlsx,.xlsx,OK,72,17,,,2022,5,,,
8,formato_tc2_20226.xlsx,.xlsx,OK,69,14,"Estrato, Ciclo, Clase de Servicio",,2022,6,,,
9,formato_tc2_20227.xlsx,.xlsx,OK,72,17,,,2022,7,,,


,archivo,formato,estado,columnas_archivo,columnas_reconocidas,faltantes,ambiguas,año_nombre,mes_nombre,separador_csv,encoding_csv,error
5,formato_tc2_20223.csv,.csv,OK,69,11,"Año de reporte, Estrato, Días Facturados, Refa...",,2022,3,"','",cp1252,
8,formato_tc2_20226.xlsx,.xlsx,OK,69,14,"Estrato, Ciclo, Clase de Servicio",,2022,6,,,


In [9]:
# ============================================================
# PROCESAMIENTO DE TODO EL HISTÓRICO
# ============================================================
# Objetivo:
#
# 1. Procesar cada archivo XLSX / CSV.
# 2. Guardar el DETALLE mensual en Parquet.
# 3. Crear un resumen NIU-periodo.
# 4. CONSERVAR variables de consumo para análisis/modelado.
# 5. No detener todo el proceso si un archivo presenta error.
#
# IMPORTANTE:
# consumo_kwh_raw todavía NO representa necesariamente el
# consumo mensual definitivo porque pueden existir:
# - múltiples facturas NIU-mes
# - refacturaciones
# - lecturas trimestrales
# ============================================================

import pandas as pd


# ============================================================
# CREAR CARPETA DE DETALLE
# ============================================================

DETAIL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# CONTENEDORES
# ============================================================

resumenes = []

errores_procesamiento = []


# ============================================================
# RECORRER TODOS LOS ARCHIVOS
# ============================================================

for numero, archivo in enumerate(
    archivos,
    start=1
):

    print(
        f"\n[{numero}/{len(archivos)}] "
        f"Procesando: {archivo.name}"
    )

    try:

        # ====================================================
        # 1. PROCESAR ARCHIVO
        # ====================================================

        df = procesar_archivo(
            archivo
        )


        # ====================================================
        # 2. VALIDACIONES BÁSICAS
        # ====================================================

        if df.empty:

            raise ValueError(
                "El archivo fue procesado "
                "pero quedó vacío."
            )


        columnas_obligatorias = [
            "NIU",
            "periodo"
        ]


        faltantes_obligatorias = [
            columna
            for columna in columnas_obligatorias
            if columna not in df.columns
        ]


        if faltantes_obligatorias:

            raise ValueError(
                "Faltan columnas necesarias "
                "después del procesamiento: "
                f"{faltantes_obligatorias}"
            )


        # ====================================================
        # 3. INFORMACIÓN DEL ARCHIVO
        # ====================================================

        print(
            f"    Filas: "
            f"{len(df):,}"
        )

        print(
            f"    Columnas: "
            f"{len(df.columns)}"
        )

        print(
            f"    NIU únicos: "
            f"{df['NIU'].nunique():,}"
        )


        if df["periodo"].notna().any():

            print(
                "    Periodo: "
                f"{df['periodo'].min():%Y-%m}"
                " → "
                f"{df['periodo'].max():%Y-%m}"
            )


        # ====================================================
        # 4. GUARDAR DETALLE MENSUAL
        # ====================================================
        #
        # Aquí se conserva la información original procesada:
        # consumo, facturas, tarifas, lecturas, etc.
        # ====================================================

        formato = (
            archivo
            .suffix
            .lower()
            .replace(".", "")
        )


        salida_detalle = (
            DETAIL_DIR
            / f"{archivo.stem}_{formato}.parquet"
        )


        df.to_parquet(
            salida_detalle,
            index=False,
            engine="pyarrow"
        )


        print(
            f"    Parquet detalle: "
            f"{salida_detalle.name}"
        )


        # ====================================================
        # 5. DEFINIR AGREGACIONES NIU - PERIODO
        # ====================================================

        agregaciones = {

            "cantidad_registros": (
                "NIU",
                "size"
            )
        }


        # ====================================================
        # CONSUMO USUARIO
        # ====================================================
        #
        # Se denomina RAW porque todavía debemos validar
        # refacturaciones y múltiples facturas.
        # ====================================================

        if "Consumo Usuario (kWh)" in df.columns:

            agregaciones[
                "consumo_kwh_raw"
            ] = (
                "Consumo Usuario (kWh)",
                lambda x: x.sum(
                    min_count=1
                )
            )


            agregaciones[
                "registros_consumo"
            ] = (
                "Consumo Usuario (kWh)",
                "count"
            )


            agregaciones[
                "consumo_kwh_min"
            ] = (
                "Consumo Usuario (kWh)",
                "min"
            )


            agregaciones[
                "consumo_kwh_max"
            ] = (
                "Consumo Usuario (kWh)",
                "max"
            )


            agregaciones[
                "consumo_kwh_mediana"
            ] = (
                "Consumo Usuario (kWh)",
                "median"
            )


        # ====================================================
        # CONSUMO PROMEDIO SEMESTRAL
        # ====================================================

        if (
            "Consumo Promedio Semestral (kWh)"
            in df.columns
        ):

            agregaciones[
                "consumo_promedio_semestral_kwh"
            ] = (
                "Consumo Promedio Semestral (kWh)",
                "median"
            )


        # ====================================================
        # TARIFA
        # ====================================================

        if (
            "Tarifa Aplicada ($/kWh)"
            in df.columns
        ):

            agregaciones[
                "tarifa_aplicada_kwh"
            ] = (
                "Tarifa Aplicada ($/kWh)",
                "median"
            )


        # ====================================================
        # REFACTURACIÓN DE CONSUMO
        # ====================================================

        if (
            "Refacturación por Consumo Usuario - (kWh)"
            in df.columns
        ):

            agregaciones[
                "refacturacion_consumo_kwh"
            ] = (
                "Refacturación por Consumo Usuario - (kWh)",
                lambda x: x.sum(
                    min_count=1
                )
            )


        # ====================================================
        # DÍAS FACTURADOS
        # ====================================================

        if "Días Facturados" in df.columns:

            agregaciones[
                "dias_facturados_max"
            ] = (
                "Días Facturados",
                "max"
            )


            agregaciones[
                "dias_facturados_mediana"
            ] = (
                "Días Facturados",
                "median"
            )


            agregaciones[
                "dias_facturados_min"
            ] = (
                "Días Facturados",
                "min"
            )


        # ====================================================
        # FECHAS DE LECTURA
        # ====================================================

        if (
            "Fecha de Lectura Actual"
            in df.columns
        ):

            agregaciones[
                "fecha_lectura_actual"
            ] = (
                "Fecha de Lectura Actual",
                "max"
            )


        if (
            "Fecha de Lectura Anterior"
            in df.columns
        ):

            agregaciones[
                "fecha_lectura_anterior"
            ] = (
                "Fecha de Lectura Anterior",
                "min"
            )


        # ====================================================
        # CICLO
        # ====================================================

        if "Ciclo" in df.columns:

            agregaciones[
                "ciclo"
            ] = (
                "Ciclo",
                "first"
            )


            agregaciones[
                "ciclos_diferentes"
            ] = (
                "Ciclo",
                "nunique"
            )


        # ====================================================
        # TIPO DE LECTURA
        # ====================================================

        if "Tipo de Lectura" in df.columns:

            agregaciones[
                "tipo_lectura"
            ] = (
                "Tipo de Lectura",
                "first"
            )


            agregaciones[
                "tipos_lectura_diferentes"
            ] = (
                "Tipo de Lectura",
                "nunique"
            )


        # ====================================================
        # TIPO FACTURA
        # ====================================================

        if "Tipo Factura" in df.columns:

            agregaciones[
                "tipo_factura"
            ] = (
                "Tipo Factura",
                "first"
            )


            agregaciones[
                "tipos_factura_diferentes"
            ] = (
                "Tipo Factura",
                "nunique"
            )


        # ====================================================
        # TIPO MEDIDOR
        # ====================================================

        if "Tipo Medidor" in df.columns:

            agregaciones[
                "tipo_medidor"
            ] = (
                "Tipo Medidor",
                "first"
            )


            agregaciones[
                "tipos_medidor_diferentes"
            ] = (
                "Tipo Medidor",
                "nunique"
            )


        # ====================================================
        # ESTRATO
        # ====================================================

        if "Estrato" in df.columns:

            agregaciones[
                "estrato"
            ] = (
                "Estrato",
                "first"
            )


        # ====================================================
        # CLASE DE SERVICIO
        # ====================================================

        if (
            "Clase de Servicio"
            in df.columns
        ):

            agregaciones[
                "clase_servicio"
            ] = (
                "Clase de Servicio",
                "first"
            )


        # ====================================================
        # 6. CREAR RESUMEN NIU - PERIODO
        # ====================================================

        resumen = (
            df
            .dropna(
                subset=[
                    "NIU",
                    "periodo"
                ]
            )
            .groupby(
                [
                    "NIU",
                    "periodo"
                ],
                as_index=False
            )
            .agg(
                **agregaciones
            )
        )


        # ====================================================
        # 7. VARIABLES DE CONTROL
        # ====================================================

        # Más de un registro para el mismo NIU-mes.
        resumen[
            "tiene_multiples_registros"
        ] = (
            resumen[
                "cantidad_registros"
            ] > 1
        )


        # Indicio de lectura aproximadamente trimestral.
        if (
            "dias_facturados_max"
            in resumen.columns
        ):

            resumen[
                "lectura_aprox_trimestral"
            ] = (
                resumen[
                    "dias_facturados_max"
                ]
                .between(
                    75,
                    105
                )
            )


        # ====================================================
        # 8. INFORMACIÓN DE ORIGEN
        # ====================================================

        resumen[
            "archivo_origen"
        ] = archivo.name

        resumen[
            "formato_origen"
        ] = archivo.suffix.lower()


        # ====================================================
        # 9. GUARDAR RESUMEN EN LISTA
        # ====================================================

        resumenes.append(
            resumen
        )


        print(
            f"    Resumen NIU-periodo: "
            f"{len(resumen):,} filas"
        )


        # Mostrar si consumo quedó incluido
        if (
            "consumo_kwh_raw"
            in resumen.columns
        ):

            print(
                "    ✓ Consumo incluido "
                "en resumen"
            )

        else:

            print(
                "    ⚠ Consumo NO disponible "
                "en este archivo"
            )


        print("    OK")


        # ====================================================
        # 10. LIBERAR MEMORIA
        # ====================================================

        del df
        del resumen


    # ========================================================
    # SI EL ARCHIVO FALLA
    # ========================================================

    except Exception as e:

        print(
            f"    ERROR: "
            f"{type(e).__name__}: "
            f"{e}"
        )


        errores_procesamiento.append({

            "archivo":
                archivo.name,

            "formato":
                archivo.suffix.lower(),

            "tipo_error":
                type(e).__name__,

            "error":
                str(e)
        })


# ============================================================
# RESUMEN FINAL
# ============================================================

print("\n")
print("=" * 60)
print("PROCESAMIENTO TERMINADO")
print("=" * 60)


print(
    f"Archivos encontrados: "
    f"{len(archivos)}"
)


print(
    f"Archivos procesados correctamente: "
    f"{len(resumenes)}"
)


print(
    f"Archivos con error: "
    f"{len(errores_procesamiento)}"
)


[1/12] Procesando: formato_tc2_20221.xlsx
Procesando: formato_tc2_20221.xlsx [.xlsx]
    Filas: 519,997
    Columnas: 24
    NIU únicos: 518,834
    Periodo: 2022-01 → 2022-01
    Parquet detalle: formato_tc2_20221_xlsx.parquet
    Resumen NIU-periodo: 518,834 filas
    ✓ Consumo incluido en resumen
    OK

[2/12] Procesando: formato_tc2_202210.xlsx
Procesando: formato_tc2_202210.xlsx [.xlsx]
    Filas: 531,436
    Columnas: 24
    NIU únicos: 530,025
    Periodo: 2022-10 → 2022-10
    Parquet detalle: formato_tc2_202210_xlsx.parquet
    Resumen NIU-periodo: 530,025 filas
    ✓ Consumo incluido en resumen
    OK

[3/12] Procesando: formato_tc2_202211.xlsx
Procesando: formato_tc2_202211.xlsx [.xlsx]
    Filas: 328,771
    Columnas: 24
    NIU únicos: 328,524
    Periodo: 2022-11 → 2022-11
    Parquet detalle: formato_tc2_202211_xlsx.parquet
    Resumen NIU-periodo: 328,524 filas
    ✓ Consumo incluido en resumen
    OK

[4/12] Procesando: formato_tc2_202212.xlsx
Procesando: formato_tc2

In [10]:
# Histórico resumido para análisis de presencia y periodicidad.
# El detalle de consumo quedó preservado en:
#   Procesado/detalle_mensual/*.parquet

historico = pd.concat(
    resumenes,
    ignore_index=True
)

historico.shape


(4701934, 30)

In [11]:
historico.to_pickle(
    OUT_DIR / "historico_temporal.pkl"
)

print("Histórico temporal guardado")

Histórico temporal guardado


In [12]:
historico["presente"] = 1

In [13]:
matriz_presencia = (
    historico
    .pivot_table(
        index="NIU",
        columns="periodo",
        values="presente",
        aggfunc="max",
        fill_value=0
    )
    .astype("uint8")
)

In [14]:
# Crear automáticamente todos los meses comprendidos entre
# el primer y el último periodo REAL cargado.
# Así el código sirve para 2022-2025, 2022-2026, etc.

periodo_min = (
    historico["periodo"]
    .min()
    .to_period("M")
    .to_timestamp()
)

periodo_max = (
    historico["periodo"]
    .max()
    .to_period("M")
    .to_timestamp()
)

meses = pd.date_range(
    start=periodo_min,
    end=periodo_max,
    freq="MS"
)

print(
    "Rango histórico:",
    periodo_min.date(),
    "a",
    periodo_max.date(),
    "| meses:",
    len(meses)
)


Rango histórico: 2022-01-01 a 2022-12-01 | meses: 12


In [15]:
matriz_presencia = matriz_presencia.reindex(
    columns=meses,
    fill_value=0
)

In [16]:
matriz_presencia.shape

(535097, 12)

In [ ]:
conteo_por_mes = (
    matriz_presencia
    .sum(axis=0)
    .rename("clientes_presentes")
    .to_frame()
)

conteo_por_mes

,clientes_presentes
2022-01-01,518834
2022-02-01,322129
2022-03-01,322527
2022-04-01,522499
2022-05-01,324139
2022-06-01,324775
2022-07-01,526230
2022-08-01,326006
2022-09-01,326817
2022-10-01,530025


In [ ]:
conteo_por_mes[
    conteo_por_mes["clientes_presentes"] > 0
]

,clientes_presentes
2022-01-01,518834
2022-02-01,322129
2022-03-01,322527
2022-04-01,522499
2022-05-01,324139
2022-06-01,324775
2022-07-01,526230
2022-08-01,326006
2022-09-01,326817
2022-10-01,530025


In [ ]:
# ============================================================
# MATRIZ DE PRESENCIA DINÁMICA
# ============================================================

# Cada NIU-periodo existente representa presencia
historico["presente"] = 1


# ------------------------------------------------------------
# Crear matriz NIU x MES
# ------------------------------------------------------------

matriz_presencia = (
    historico
    .pivot_table(
        index="NIU",
        columns="periodo",
        values="presente",
        aggfunc="max",
        fill_value=0
    )
)


# ------------------------------------------------------------
# Crear automáticamente todos los meses entre
# el primer y último archivo disponible
# ------------------------------------------------------------

periodo_min = (
    historico["periodo"]
    .min()
    .to_period("M")
    .to_timestamp()
)

periodo_max = (
    historico["periodo"]
    .max()
    .to_period("M")
    .to_timestamp()
)


meses = pd.date_range(
    start=periodo_min,
    end=periodo_max,
    freq="MS"
)


# ------------------------------------------------------------
# Completar meses faltantes
# ------------------------------------------------------------

matriz_presencia = (
    matriz_presencia
    .reindex(
        columns=meses,
        fill_value=0
    )
    .astype("uint8")
)


print(
    "Periodo analizado:",
    periodo_min,
    "→",
    periodo_max
)

print(
    "Dimensiones:",
    matriz_presencia.shape
)

Periodo analizado: 2022-01-01 00:00:00 → 2022-12-01 00:00:00
Dimensiones: (535097, 12)


In [ ]:
def crear_matriz_anual(
    historico,
    año
):

    datos = historico[
        historico["periodo"].dt.year == año
    ].copy()

    datos["presente"] = 1

    matriz = (
        datos
        .pivot_table(
            index="NIU",
            columns="periodo",
            values="presente",
            aggfunc="max",
            fill_value=0
        )
    )

    meses_año = pd.date_range(
        start=f"{año}-01-01",
        end=f"{año}-12-01",
        freq="MS"
    )

    matriz = (
        matriz
        .reindex(
            columns=meses_año,
            fill_value=0
        )
        .astype("uint8")
    )

    return matriz

In [ ]:
matriz_2022 = crear_matriz_anual(
    historico,
    2022
)

In [ ]:
meses_por_cliente_2022 = (
    matriz_2022.sum(axis=1)
)

meses_por_cliente_2022 \
    .value_counts() \
    .sort_index()

1       5350
2       2814
3       2729
4     198226
5        791
6        637
7        842
8        832
9        899
10       638
11      1001
12    320338
Name: count, dtype: int64

In [ ]:
# ============================================================
# CREAR MATRIZ DE PRESENCIA PARA UN AÑO
# ============================================================

def crear_matriz_anual(historico, año):

    datos = historico[
        historico["periodo"].dt.year == año
    ].copy()

    if datos.empty:
        raise ValueError(
            f"No existen datos para el año {año}"
        )

    datos["presente"] = 1

    # Meses completos del año
    meses_año = pd.date_range(
        start=f"{año}-01-01",
        end=f"{año}-12-01",
        freq="MS"
    )

    matriz = (
        datos
        .pivot_table(
            index="NIU",
            columns="periodo",
            values="presente",
            aggfunc="max",
            fill_value=0
        )
        .reindex(
            columns=meses_año,
            fill_value=0
        )
        .astype("uint8")
    )

    return matriz

In [ ]:
def analizar_apariciones_anuales(historico, año):

    matriz = crear_matriz_anual(
        historico,
        año
    )

    meses_por_cliente = (
        matriz.sum(axis=1)
    )

    resumen = (
        meses_por_cliente
        .value_counts()
        .sort_index()
        .rename_axis("meses_presentes")
        .reset_index(name="cantidad_clientes")
    )

    resumen["porcentaje"] = (
        resumen["cantidad_clientes"]
        / len(matriz)
        * 100
    )

    return matriz, resumen

In [ ]:
matriz_año, resumen_apariciones = (
    analizar_apariciones_anuales(
        historico,
        2022
    )
)

display(resumen_apariciones)

,meses_presentes,cantidad_clientes,porcentaje
0,1,5350,0.999819
1,2,2814,0.525886
2,3,2729,0.510001
3,4,198226,37.044872
4,5,791,0.147824
5,6,637,0.119044
6,7,842,0.157355
7,8,832,0.155486
8,9,899,0.168007
9,10,638,0.119231


In [ ]:
clientes_4_meses = matriz_año[
    matriz_año.sum(axis=1) == 4
]

print(
    f"Clientes con 4 meses: "
    f"{len(clientes_4_meses):,}"
)

clientes_4_meses.head(20)

Clientes con 4 meses: 198,226


,2022-01-01,2022-02-01,2022-03-01,2022-04-01,2022-05-01,2022-06-01,2022-07-01,2022-08-01,2022-09-01,2022-10-01,2022-11-01,2022-12-01
NIU,,,,,,,,,,,,
100000949,1,0,0,1,0,0,1,0,0,1,0,0
100001726,1,0,0,1,0,0,1,0,0,1,0,0
100002503,1,0,0,1,0,0,1,0,0,1,0,0
100003390,1,0,0,1,0,0,1,0,0,1,0,0
100004177,1,0,0,1,0,0,1,0,0,1,0,0
100005854,1,0,0,1,0,0,1,0,0,1,0,0
100006631,1,0,0,1,0,0,1,0,0,1,0,0
100007418,1,0,0,1,0,0,1,0,0,1,0,0
100008205,1,0,0,1,0,0,1,0,0,1,0,0


In [ ]:
print(
    clientes_4_meses.columns.min()
)

print(
    clientes_4_meses.columns.max()
)

2022-01-01 00:00:00
2022-12-01 00:00:00


In [ ]:
# ============================================================
# EXPLORACIÓN DE ARCHIVOS PARQUET
# ============================================================

from IPython.display import display, HTML
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


def explorar_parquet(
    ruta,
    nombre=None,
    cargar_completo=True,
    mostrar_muestra=True,
    n_muestra=5
):
    """
    Explora un archivo Parquet.

    Parámetros
    ----------
    ruta : str o Path
        Ruta del archivo parquet.

    nombre : str, opcional
        Nombre que se mostrará en el reporte.

    cargar_completo : bool
        True  -> carga todo el Parquet y realiza análisis completo.
        False -> solo muestra metadatos y una muestra.

    mostrar_muestra : bool
        Mostrar primeros registros.

    n_muestra : int
        Cantidad de registros a mostrar.
    """

    ruta = Path(ruta)

    if not ruta.exists():
        raise FileNotFoundError(
            f"No existe el archivo:\n{ruta}"
        )

    if ruta.suffix.lower() != ".parquet":
        raise ValueError(
            f"El archivo no es Parquet: {ruta.name}"
        )

    if nombre is None:
        nombre = ruta.name

    # ========================================================
    # 1. LEER METADATOS SIN CARGAR TODO EL ARCHIVO
    # ========================================================

    parquet = pq.ParquetFile(ruta)

    metadata = parquet.metadata

    filas = metadata.num_rows
    columnas = metadata.num_columns
    row_groups = metadata.num_row_groups

    tamaño_mb = (
        ruta.stat().st_size
        / 1024
        / 1024
    )

    # ========================================================
    # 2. ENCABEZADO
    # ========================================================

    display(
        HTML(
            f"""
            <div style="
                background:#1F3864;
                color:white;
                padding:12px 20px;
                border-radius:8px;
                margin:10px 0;
                font-size:18px;
                font-weight:bold;
            ">
                🔍 Exploración Parquet: {nombre}
            </div>
            """
        )
    )

    # ========================================================
    # 3. TARJETAS GENERALES
    # ========================================================

    display(
        HTML(
            f"""
            <div style="
                display:flex;
                gap:12px;
                margin:10px 0;
                flex-wrap:wrap;
            ">

                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #2E75B6;
                ">
                    <div style="font-size:12px;color:#666;">
                        FILAS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {filas:,}
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #2E75B6;
                ">
                    <div style="font-size:12px;color:#666;">
                        COLUMNAS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {columnas}
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #1D9E75;
                ">
                    <div style="font-size:12px;color:#666;">
                        TAMAÑO
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {tamaño_mb:,.1f} MB
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #F4A300;
                ">
                    <div style="font-size:12px;color:#666;">
                        ROW GROUPS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {row_groups}
                    </div>
                </div>

            </div>
            """
        )
    )

    # ========================================================
    # 4. ESQUEMA PARQUET
    # ========================================================

    print("\n📐 Esquema del Parquet:")

    schema = parquet.schema_arrow

    esquema = pd.DataFrame({
        "Columna": schema.names,
        "Tipo Parquet": [
            str(campo.type)
            for campo in schema
        ]
    })

    display(esquema)


    # ========================================================
    # 5. MODO LIGERO
    # ========================================================

    if not cargar_completo:

        print(
            "\n⚡ Modo ligero: "
            "no se cargará todo el archivo en memoria."
        )

        if mostrar_muestra:

            # Leer primer row group
            muestra = (
                parquet
                .read_row_group(0)
                .to_pandas()
                .head(n_muestra)
            )

            print(
                f"\n📋 Primeros "
                f"{n_muestra} registros:"
            )

            display(muestra)

        return None


    # ========================================================
    # 6. CARGAR PARQUET COMPLETO
    # ========================================================

    print(
        "\n📥 Cargando archivo completo..."
    )

    df = pd.read_parquet(
        ruta,
        engine="pyarrow"
    )

    print("✓ Archivo cargado")


    # ========================================================
    # 7. TIPOS DE VARIABLES
    # ========================================================

    n_num = len(
        df.select_dtypes(
            include="number"
        ).columns
    )

    n_cat = len(
        df.select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        ).columns
    )

    n_fecha = len(
        df.select_dtypes(
            include=[
                "datetime",
                "datetimetz"
            ]
        ).columns
    )


    duplicados = df.duplicated().sum()


    display(
        HTML(
            f"""
            <div style="
                display:flex;
                gap:12px;
                margin:10px 0;
                flex-wrap:wrap;
            ">

                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #1D9E75;
                ">
                    <div style="font-size:12px;color:#666;">
                        NUMÉRICAS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {n_num}
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #F4A300;
                ">
                    <div style="font-size:12px;color:#666;">
                        CATEGÓRICAS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {n_cat}
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #7030A0;
                ">
                    <div style="font-size:12px;color:#666;">
                        FECHAS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {n_fecha}
                    </div>
                </div>


                <div style="
                    background:#EAF1F8;
                    padding:12px 20px;
                    border-radius:8px;
                    border-left:4px solid #C0504D;
                ">
                    <div style="font-size:12px;color:#666;">
                        DUPLICADOS
                    </div>

                    <div style="
                        font-size:22px;
                        font-weight:bold;
                        color:#1F3864;
                    ">
                        {duplicados:,}
                    </div>
                </div>

            </div>
            """
        )
    )


    # ========================================================
    # 8. PRIMEROS REGISTROS
    # ========================================================

    if mostrar_muestra:

        print(
            f"\n📋 Primeros "
            f"{n_muestra} registros:"
        )

        display(
            df.head(n_muestra)
        )


    # ========================================================
    # 9. RESUMEN DE COLUMNAS
    # ========================================================

    print(
        "\n📊 Resumen de columnas:"
    )

    resumen = pd.DataFrame({

        "Tipo":
            df.dtypes.astype(str),

        "Nulos":
            df.isna().sum(),

        "% Nulos":
            (
                df.isna().mean()
                * 100
            ).round(2),

        "Únicos":
            df.nunique(
                dropna=True
            )
    })


    estilo_th = [
        {
            "selector": "th",
            "props": [
                (
                    "background-color",
                    "#1F3864"
                ),
                (
                    "color",
                    "white"
                ),
                (
                    "font-weight",
                    "bold"
                ),
                (
                    "text-align",
                    "left"
                )
            ]
        }
    ]


    display(
        resumen.style
        .format({
            "% Nulos":
                "{:.2f}%"
        })
        .background_gradient(
            subset=["% Nulos"],
            cmap="Reds",
            vmin=0,
            vmax=100
        )
        .bar(
            subset=["Únicos"],
            color="#7FB3D5"
        )
        .set_table_styles(
            estilo_th
        )
    )


    # ========================================================
    # 10. DUPLICADOS
    # ========================================================

    print(
        "\n🔁 Análisis de duplicados:"
    )

    pct_dup = (
        duplicados
        / len(df)
        * 100
        if len(df) > 0
        else 0
    )

    print(
        f"  • Filas idénticas: "
        f"{duplicados:,} "
        f"({pct_dup:.2f}%)"
    )


    # ========================================================
    # 11. DUPLICADOS POR NIU
    # ========================================================

    if "NIU" in df.columns:

        niu_unicos = (
            df["NIU"]
            .nunique()
        )

        print(
            f"  • NIU únicos: "
            f"{niu_unicos:,}"
        )

        registros_niu_repetidos = (
            df["NIU"]
            .duplicated(
                keep=False
            )
            .sum()
        )

        print(
            f"  • Registros pertenecientes "
            f"a NIU repetidos: "
            f"{registros_niu_repetidos:,}"
        )


    # ========================================================
    # 12. DUPLICADOS NIU + PERIODO
    # ========================================================

    if (
        "NIU" in df.columns
        and
        "periodo" in df.columns
    ):

        duplicados_niu_periodo = (
            df.duplicated(
                subset=[
                    "NIU",
                    "periodo"
                ]
            )
            .sum()
        )

        print(
            f"  • Duplicados NIU-periodo: "
            f"{duplicados_niu_periodo:,}"
        )


    # ========================================================
    # 13. ESTADÍSTICAS NUMÉRICAS
    # ========================================================

    df_num = df.select_dtypes(
        include="number"
    )

    if not df_num.empty:

        print(
            "\n🔢 Estadísticas descriptivas:"
        )

        display(
            df_num
            .describe()
            .T
            .style
            .format("{:,.2f}")
            .background_gradient(
                subset=["std"],
                cmap="Oranges"
            )
            .set_table_styles(
                estilo_th
            )
        )


    # ========================================================
    # 14. CATEGÓRICAS
    # ========================================================

    cols_cat = (
        df.select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
    )

    if len(cols_cat) > 0:

        print(
            "\n🏷️ Variables categóricas:"
        )

        resumen_cat = pd.DataFrame({

            "Nulos":
                df[cols_cat]
                .isna()
                .sum(),

            "Únicos":
                df[cols_cat]
                .nunique(),

            "Moda":
                [
                    (
                        df[col]
                        .mode()
                        .iloc[0]
                        if not df[col]
                        .mode()
                        .empty
                        else pd.NA
                    )
                    for col in cols_cat
                ]
        })

        display(
            resumen_cat
        )


        print(
            "\n🏷️ Muestra de valores:"
        )

        for col in cols_cat:

            valores = (
                df[col]
                .dropna()
                .unique()[:8]
            )

            muestra = ", ".join(
                str(valor)
                for valor in valores
            )

            cantidad_unicos = (
                df[col]
                .nunique()
            )

            extra = (
                f" ... "
                f"(+{cantidad_unicos - 8:,} más)"
                if cantidad_unicos > 8
                else ""
            )

            print(
                f"  • {col}: "
                f"{muestra}"
                f"{extra}"
            )


    # ========================================================
    # DEVOLVER DATAFRAME
    # ========================================================

    return df

In [ ]:
# ============================================================
# CONVERTIR HISTÓRICO 2022 DE PICKLE A PARQUET
# ============================================================

ruta_pickle = (
    OUT_DIR
    / "historico_2022.pkl"
)

ruta_parquet = (
    OUT_DIR
    / "historico_2022.parquet"
)

# Cargar Pickle
historico_2022 = pd.read_pickle(
    ruta_pickle
)

# Guardar como Parquet
historico_2022.to_parquet(
    ruta_parquet,
    index=False,
    engine="pyarrow"
)

print("Parquet generado correctamente:")
print(ruta_parquet)

Parquet generado correctamente:
C:\Users\Home\Documents\Datos Ebsa\Procesado\historico_2022.parquet


In [ ]:
df_2022 = explorar_parquet(
    OUT_DIR / "historico_2022.parquet",
    nombre="Histórico TC2 - 2022"
)


📐 Esquema del Parquet:


,Columna,Tipo Parquet
0,NIU,large_string
1,periodo,timestamp[us]
2,cantidad_registros,int64
3,dias_facturados_max,double
4,dias_facturados_mediana,double
5,fecha_lectura_actual,timestamp[us]
6,fecha_lectura_anterior,timestamp[us]
7,ciclo,double
8,ciclos_diferentes,double
9,tipo_lectura,int64



📥 Cargando archivo completo...
✓ Archivo cargado



📋 Primeros 5 registros:


,NIU,periodo,cantidad_registros,dias_facturados_max,dias_facturados_mediana,fecha_lectura_actual,fecha_lectura_anterior,ciclo,ciclos_diferentes,tipo_lectura,tipo_factura,tipo_medidor,estrato,clase_servicio,archivo_origen,formato_origen
0,1000006589,2022-01-01,1,31.0,31.0,2022-01-20,2021-12-20,1.0,1.0,1,1,1,2.0,RS,formato_tc2_20221.xlsx,.xlsx
1,1000007366,2022-01-01,1,31.0,31.0,2022-01-17,2021-12-17,1.0,1.0,1,1,1,2.0,RS,formato_tc2_20221.xlsx,.xlsx
2,1000008143,2022-01-01,1,29.0,29.0,2022-01-19,2021-12-21,1.0,1.0,1,1,1,3.0,RS,formato_tc2_20221.xlsx,.xlsx
3,100000949,2022-01-01,1,97.0,97.0,2022-01-16,2021-10-11,12.0,1.0,1,1,1,2.0,RS,formato_tc2_20221.xlsx,.xlsx
4,1000009920,2022-01-01,1,29.0,29.0,2022-01-19,2021-12-21,1.0,1.0,1,1,1,3.0,RS,formato_tc2_20221.xlsx,.xlsx


...


🔁 Análisis de duplicados:
  • Filas idénticas: 0 (0.00%)
  • NIU únicos: 535,097
  • Registros pertenecientes a NIU repetidos: 4,696,584
  • Duplicados NIU-periodo: 0

🔢 Estadísticas descriptivas:


,count,mean,std,min,25%,50%,75%,max
cantidad_registros,"4,701,934.00",1.00,0.09,1.00,1.00,1.00,1.00,65.00
dias_facturados_max,"4,379,407.00",41.73,23.49,0.00,30.00,31.00,32.00,142.00
dias_facturados_mediana,"4,379,407.00",41.73,23.49,0.00,30.00,31.00,32.00,142.00
ciclo,"4,054,632.00",5.18,6.58,0.00,1.00,2.00,9.00,90.00
ciclos_diferentes,"4,054,632.00",1.00,0.00,1.00,1.00,1.00,1.00,2.00
tipo_lectura,"4,701,934.00",1.01,0.13,1.00,1.00,1.00,1.00,3.00
tipo_factura,"4,701,934.00",1.00,0.00,1.00,1.00,1.00,1.00,1.00
tipo_medidor,"4,701,934.00",1.12,0.38,1.00,1.00,1.00,1.00,6.00
estrato,"4,054,594.00",2.31,0.84,1.00,2.00,2.00,3.00,6.00



🏷️ Variables categóricas:


,Nulos,Únicos,Moda
NIU,0,535097,1000006589
clase_servicio,647340,11,RS
archivo_origen,0,12,formato_tc2_202210.xlsx
formato_origen,0,2,.xlsx



🏷️ Muestra de valores:
  • NIU: 1000006589, 1000007366, 1000008143, 100000949, 1000009920, 1000010766, 100001726, 100002503 ... (+535,089 más)
  • clase_servicio: RS, CR, OF, ID, AC, RI, AA, PR ... (+3 más)
  • archivo_origen: formato_tc2_20221.xlsx, formato_tc2_202210.xlsx, formato_tc2_202211.xlsx, formato_tc2_202212.xlsx, formato_tc2_20222.xlsx, formato_tc2_20223.csv, formato_tc2_20224.xlsx, formato_tc2_20225.xlsx ... (+4 más)
  • formato_origen: .xlsx, .csv
